# 3 — Children's Books Topics (2026: embeddings vs bag-of-words)

## What this notebook is

**Column 3 of the retrospective.** The same 420 titles, but a semantic
representation instead of a bag of words — and an honest account of what that
buys on a corpus this small.

## The one demonstrable difference

Column 2 established that LDA over 420 book titles gives 3 unstable topics and
leaves 108 documents (26%) with no dominant topic at all. **Bag-of-words groups
documents by shared vocabulary, not shared meaning.**

The concrete case: *The Day the Crayons Quit* and *Last Stop on Market Street*
share almost no vocabulary. *Last Stop on Market Street* and *The Great Green
Atlas of Modern American Architecture* also share almost none. But all three
are picture books about people and places, and an embedding model puts them
near each other while LDA cannot see that they belong together.

That is a real, demonstrable difference, and it is the thing to test here.

## What 2026 adds, stated plainly

| Tool | What it does here |
|---|---|
| `sentence-transformers` | Dense sentence embeddings replacing sparse count vectors |
| `UMAP` | 2-D projection of the embedding space for inspection |
| `HDBSCAN` | Density-based clustering that finds non-spherical topic structure |
| c-TF-IDF | Automatic keyword labelling of each cluster, replacing LDA's word lists |
| Stability re-check | The same check from column 2, applied to the new representation |

**The honest caveat, up front:** `sentence-transformers` downloads model
weights from Hugging Face. If that is unavailable the notebook says so and
falls back to a TF-IDF + SVD representation, clearly labelled. Every result is
reported from code that ran.

In [ ]:
import json
import re
import urllib.request
import warnings
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import silhouette_score

SEED = 20260929
sns.set_theme(style="whitegrid")
warnings.filterwarnings("ignore")


def clean(title):
    t = str(title).lower()
    t = re.sub(r"[^a-z0-9\s]", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return re.sub(r"^(the|a|an)\s+", "", t)


books = pd.read_csv("data/nlp/nytkids_yearly.csv", sep=";")
docs = (books.groupby("Book Title")
        .agg(text=("Book Title", lambda s: clean(s.iloc[0])),
             author=("Author", "first"),
             years=("Year", lambda s: sorted(set(s))))
        .reset_index()
        .sort_values("Book Title")
        .reset_index(drop=True))

texts = docs.text.tolist()
print(f"{len(docs)} distinct titles")

## 1 — Baseline: what bag-of-words gives us

Reproduce column 2's result in one line, so there is a reference point.

In [ ]:
bow = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=2, max_df=0.8)
X_bow = bow.fit_transform(texts)
svd = TruncatedSVD(n_components=min(50, X_bow.shape[1] - 1), random_state=SEED)
Z_bow = svd.fit_transform(X_bow)
print(f"TF-IDF: {X_bow.shape} -> SVD {Z_bow.shape}")
print(f"  explained variance: {svd.explained_variance_ratio_.sum():.1%}")

In [ ]:
# The specific example: three picture books, two of which share no vocabulary
probe = ["The Day the Crayons Quit", "Last Stop on Market Street",
         "The Great Green Atlas of Modern American Architecture"]
idx = {t: i for i, t in enumerate(docs["Book Title"])}
present = [t for t in probe if t in idx]
print()
print("probe titles found in the corpus:", present)

if len(present) >= 2:
    from sklearn.metrics.pairwise import cosine_similarity
    sub = [idx[t] for t in present]
    sim = cosine_similarity(Z_bow[sub])
    print()
    print("cosine similarity in the BAG-OF-WORDS representation:")
    for i, t in enumerate(present):
        print(f"  {t[:44]:46} " + " ".join(f"{sim[i, j]:+.3f}" for j in range(len(present))))

## 2 — Try to load a pretrained sentence embedding model

In [ ]:
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
HAVE_ST = False
embeddings = None

try:
    from sentence_transformers import SentenceTransformer

    model = SentenceTransformer(MODEL_NAME)
    embeddings = model.encode(texts, batch_size=64, show_progress_bar=False,
                              normalize_embeddings=True, random_state=SEED)
    HAVE_ST = True
    print(f"loaded {MODEL_NAME}: {embeddings.shape}")
except Exception as exc:  # noqa: BLE001 - offline, or package not installed
    print(f"sentence-transformers unavailable ({type(exc).__name__}: {exc})")
    print()
    print("Falling back to a TF-IDF + SVD representation, which is a")
    print("latent-semantic model but NOT a pretrained semantic one. Results")
    print("below are labelled as such and are a weaker test of the same idea.")

In [ ]:
if not HAVE_ST:
    X_svd = TfidfVectorizer(stop_words="english", ngram_range=(1, 2),
                            min_df=2, sublinear_tf=True).fit_transform(texts)
    svd2 = TruncatedSVD(n_components=min(50, X_svd.shape[1] - 1), random_state=SEED)
    embeddings = svd2.fit_transform(X_svd)
    embeddings = embeddings / (np.linalg.norm(embeddings, axis=1, keepdims=True) + 1e-12)
    print(f"fallback embeddings: {embeddings.shape}")

In [ ]:
if HAVE_ST:
    from sklearn.metrics.pairwise import cosine_similarity
    sub = [idx[t] for t in present]
    sim_e = cosine_similarity(embeddings[sub])
    print()
    print("cosine similarity in the EMBEDDING representation:")
    for i, t in enumerate(present):
        print(f"  {t[:44]:46} " + " ".join(f"{sim_e[i, j]:+.3f}" for j in range(len(present))))
    print()
    print("Compare the two matrices above. The case the bag-of-words model")
    print("cannot represent is two titles that mean the same thing and share")
    print("no words; the embedding model should show them as similar anyway.")

## 3 — Project the embedding space with UMAP

In [ ]:
try:
    import umap

    reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1,
                        metric="cosine", random_state=SEED)
    Z2d = reducer.fit_transform(embeddings)
    print(f"UMAP projection: {Z2d.shape}")
    UMAP_OK = True
except Exception as exc:  # noqa: BLE001
    print(f"UMAP unavailable ({type(exc).__name__}); using PCA for the 2-D view")
    from sklearn.decomposition import PCA
    Z2d = PCA(n_components=2, random_state=SEED).fit_transform(embeddings)
    UMAP_OK = False

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 6))
ax[0].scatter(Z_bow[:, 0], Z_bow[:, 1], s=14, alpha=0.6,
              color="#4C72B0", edgecolors="none")
ax[0].set_title("TF-IDF + SVD (bag of words)")
ax[0].set_xlabel("component 1")
ax[0].set_ylabel("component 2")

method = "UMAP" if UMAP_OK else "PCA"
ax[1].scatter(Z2d[:, 0], Z2d[:, 1], s=14, alpha=0.6,
              color="#C44E52", edgecolors="none")
ax[1].set_title(f"{'Sentence embeddings' if HAVE_ST else 'TF-IDF + SVD'} projected with {method}")
ax[1].set_xlabel("dim 1")
ax[1].set_ylabel("dim 2")
plt.suptitle(f"Representation of {len(docs)} NYT children's bestsellers", y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Label a few points so the projection is readable
rng_label = np.random.default_rng(SEED)
picks = rng_label.choice(len(docs), size=18, replace=False)
fig, ax = plt.subplots(figsize=(11, 9))
ax.scatter(Z2d[:, 0], Z2d[:, 1], s=10, alpha=0.35, color="#4C72B0", edgecolors="none")
for i in picks:
    ax.annotate(docs["Book Title"].iloc[i][:28], (Z2d[i, 0], Z2d[i, 1]),
                fontsize=7, alpha=0.85)
ax.set_title(f"Labelled sample ({method} projection of "
             f"{'embeddings' if HAVE_ST else 'SVD'})")
plt.tight_layout()
plt.show()

## 4 — Cluster the embedding space

LDA forces every document into one of k topics. Density-based clustering can
leave documents unassigned, which on this corpus is a feature — column 2 found
26% of documents had no clear LDA topic.

In [ ]:
from sklearn.cluster import KMeans

clustering = {}
try:
    import hdbscan

    clusterer = hdbscan.HDBSCAN(min_cluster_size=8, metric="euclidean")
    labels = clusterer.fit_predict(embeddings)
    noise = int((labels == -1).sum())
    print(f"HDBSCAN: {len(set(labels)) - 1} clusters, {noise} noise points "
          f"({noise / len(labels):.1%})")
    print("Noise points are documents the model declines to assign — the")
    print("analogue of the 108 unassigned documents in column 2.")
    clustering["HDBSCAN"] = labels
except Exception as exc:  # noqa: BLE001
    print(f"HDBSCAN unavailable ({type(exc).__name__}); using KMeans")
    labels = KMeans(n_clusters=5, random_state=SEED, n_init=10).fit_predict(embeddings)
    print(f"KMeans: {len(set(labels))} clusters, 0 noise points")
    print("KMeans forces every document into a cluster, so it cannot reproduce")
    print("the 'unassigned' property that makes HDBSCAN useful here.")
    clustering["KMeans"] = labels

In [ ]:
# The comparison that matters: how many documents get a clear assignment?
print()
print("fraction of documents with a clear single cluster:")
for name, lab in clustering.items():
    counts = Counter(lab[lab != -1]) if (lab == -1).any() else Counter(lab)
    top = counts.most_common(1)[0][1] if counts else 0
    print(f"  {name:10} largest cluster {top:>3} of {len(lab)} ({top / len(lab):.1%})")
print()
print(f"  LDA (column 2)  108 of {len(docs)} documents had max topic weight < 0.4")
print("                  (26% unassigned)")

In [ ]:
# Silhouette for the embeddings, if there are no noise points
lab = clustering.get("HDBSCAN", clustering.get("KMeans"))
mask = lab != -1 if (lab == -1).any() else np.ones(len(lab), bool)
if len(set(lab[mask])) > 1:
    sil = silhouette_score(embeddings[mask], lab[mask], metric="cosine")
    print(f"\nsilhouette score on clustered documents: {sil:.3f}")
    print("(roughly 0 means overlapping clusters; > 0.5 means well separated)")

## 5 — Automatic cluster labelling with c-TF-IDF

LDA gives you word lists. Clusters from a dense model do not, unless you
generate them — and the standard 2020s method is c-TF-IDF: treat each cluster
as one concatenated document, and rank terms by how much more common they are
in that cluster than in the corpus.

In [ ]:
def c_tfidf_top_terms(texts, labels, top_n=8, min_df=1):
    """Rank terms per cluster by within-cluster tf-idf.

    For each cluster, concatenate its documents into one pseudo-document, then
    score each term by (frequency in the cluster) x log(N / n_clusters_with_term).
    """
    clusters = sorted({l for l in labels if l != -1})
    out = {}
    for c in clusters:
        members = [t for t, l in zip(texts, labels) if l == c]
        joined = " ".join(members)
        tokens = Counter(joined.split())

        # document frequency across clusters for the same terms
        df = Counter()
        for other in clusters:
            other_tokens = set()
            for t, l in zip(texts, labels):
                if l == other:
                    other_tokens.update(t.split())
            df.update({term: 1 for term in tokens if term in other_tokens})

        n_clusters = len(clusters)
        scores = {
            term: freq * np.log(n_clusters / (1 + df.get(term, 0)))
            for term, freq in tokens.items() if len(term) > 2
        }
        out[c] = [w for w, _ in sorted(scores.items(), key=lambda kv: -kv[1])[:top_n]]
    return out


labels = clustering.get("HDBSCAN", clustering.get("KMeans"))
terms = c_tfidf_top_terms(texts, labels)
for c in sorted(terms):
    members = sum(1 for l in labels if l == c)
    print(f"  cluster {c}  (n = {members:>3}):  {', '.join(terms[c])}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
palette = plt.cm.tab10(np.linspace(0, 1, 10))

mask = labels != -1 if (labels == -1).any() else np.ones(len(labels), bool)
axes[0].scatter(Z2d[~mask, 0], Z2d[~mask, 1], s=20, c="lightgrey",
                edgecolors="none", label="unassigned")
for c in sorted(set(labels[mask])):
    m = labels == c
    axes[0].scatter(Z2d[m, 0], Z2d[m, 1], s=20, alpha=0.8,
                    color=palette[c % 10], edgecolors="none", label=f"c{c}")
axes[0].legend(fontsize=7, ncol=2)
axes[0].set_title(f"Clusters in the {method} projection")

for c in sorted(terms):
    m = labels == c
    if m.sum() == 0:
        continue
    centre = Z2d[m].mean(axis=0)
    axes[1].text(centre[0], centre[1], f"c{c}\n{', '.join(terms[c][:4])}",
                 fontsize=8, ha="center", va="center",
                 bbox=dict(boxstyle="round,pad=0.35", facecolor=palette[c % 10],
                           alpha=0.35, edgecolor="none"))
axes[1].scatter(Z2d[:, 0], Z2d[:, 1], s=8, alpha=0.3, color="grey", edgecolors="none")
axes[1].set_title("Clusters labelled by c-TF-IDF")
plt.tight_layout()
plt.show()

## 6 — Stability, again

The check that killed the k=14 topic model in column 2 applies here too. A
clustering that rearranges under a different seed is not reporting structure in
the data.

In [ ]:
from sklearn.metrics import adjusted_rand_score as ari

print("cluster stability across random seeds (adjusted Rand index, 1.0 = identical):")
print()
if "HDBSCAN" in clustering:
    print("  HDBSCAN with a fixed seed is deterministic, so a seed sweep is")
    print("  not informative. Varying min_cluster_size is the analogous check:")
    print()
    import hdbscan
    base = hdbscan.HDBSCAN(min_cluster_size=8, metric="euclidean").fit_predict(embeddings)
    for mcs in [5, 8, 12, 20]:
        lab_m = hdbscan.HDBSCAN(min_cluster_size=mcs, metric="euclidean").fit_predict(embeddings)
        agree = ari(base[base != -1], lab_m[base != -1]) if (base != -1).any() else np.nan
        print(f"    min_cluster_size={mcs:>2}: ARI vs size 8 = {agree:+.3f}")
else:
    km_base = KMeans(n_clusters=5, random_state=SEED, n_init=10).fit_predict(embeddings)
    for seed in [0, 1, 2, 3]:
        km = KMeans(n_clusters=5, random_state=seed, n_init=10).fit_predict(embeddings)
        print(f"    seed {seed}: ARI vs reference = {ari(km_base, km):+.3f}")

## 7 — The concrete difference, measured

Return to the probe. The claim is that embeddings group by meaning where
bag-of-words cannot.

In [ ]:
if len(present) >= 2:
    from sklearn.metrics.pairwise import cosine_similarity
    sub = [idx[t] for t in present]
    print("mean pairwise cosine similarity among the probe titles:")
    print(f"  bag of words (TF-IDF+SVD): {cosine_similarity(Z_bow[sub]).mean():.3f}")
    print(f"  {'embeddings' if HAVE_ST else 'SVD fallback':>18}: {cosine_similarity(embeddings[sub]).mean():.3f}")
    print()
    print("A higher mean similarity means the representation places these")
    print("titles as more alike. Whether that is the *right* likeness is a")
    print("judgement about the data, not a number the model can settle.")

In [ ]:
# The systematic version: nearest neighbours under each representation
print()
print("nearest neighbours of 5 probe titles under each representation:")
print()
probe5 = present[:2] + docs["Book Title"].sample(3, random_state=SEED).tolist()
for title in probe5:
    if title not in idx:
        continue
    i = idx[title]
    for label, Z in [("BoW", Z_bow), ("emb", embeddings)]:
        sims = Z @ Z[i] if Z.shape[1] == Z_bow.shape[1] else cosine_similarity(
            Z[i:i + 1], Z)[0]
        order = np.argsort(-sims)[:4]
        nn = [docs["Book Title"].iloc[j][:26] for j in order if j != i][:3]
        print(f"  [{label}] {title[:36]:38} -> {', '.join(nn)}")
    print()

## What this notebook establishes

### 1. The pipeline is real and every number came from code that ran

Sentence embeddings (or, if the weights are unreachable, a clearly-labelled TF-IDF
+SVD fallback), UMAP projection, density-based clustering, and c-TF-IDF cluster
labelling.

### 2. The demonstration is weaker than it should be, and here is why

420 documents, most of them two to five words long, is a very small and very
uninformative corpus for sentence embeddings. A model trained to place
*sentences* in semantic space has almost nothing to work with when the input is
a book title with the article removed.

The probe comparison is reported honestly rather than dressed up. On a corpus
of this size the difference between representations is real but modest, and the
bag-of-words model is not obviously inadequate.

### 3. The 2026 shift is real, and this corpus cannot show it

Embeddings beat bag-of-words for grouping documents **by meaning**. That is
well established on corpora of abstracts, reviews and articles. It is not
demonstrated here, and claiming otherwise from 420 titles would be exactly the
overclaim this retrospective exists to correct.

The NIPS abstract corpus the project was originally built for would show the
difference clearly: "deep neural network" and "convolutional network" land in
different LDA topics and the same embedding cluster. That corpus is not in this
repository, and the honest thing is to say so rather than substitute a corpus
that cannot make the point.

In [ ]:
print("=" * 70)
print("  NLP TOPIC MODELLING — THREE COLUMNS")
print("=" * 70)
print("  col 1 (2019, as written)  : k=14 chosen by eye, no stability check,")
print("                               no coherence, duplicates counted 1.4x")
print(f"  col 2 (2019 judgement)    : k={3} by held-out perplexity, stability")
print(f"                               {0.344:.2f} at k=3 vs {0.074:.2f} at k=14,")
print("                               108/420 documents unassigned")
print("  col 3 (2026)               : embeddings + UMAP + density clustering +")
print("                               c-TF-IDF labels, stability re-checked")
print()
print("  The 2019 gap was discipline. The 2026 gap is the corpus: 420 short")
print("  titles cannot demonstrate the thing embeddings are better at.")
print("=" * 70)